# Let's go PRO!

Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [1]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

load_dotenv(override = True)

MODEL = "gpt-4.1-nano"

DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

openai = OpenAI()

In [2]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [3]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description = "A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description = "A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description = "The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content = self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata = metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

That's it!

### Let's start with Step 1

In [4]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding = "utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [5]:
documents = fetch_documents()

Loaded 76 documents


### Donezo! On to Step 2 - make the chunks

In [6]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [7]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: company
The document has been retrieved from: knowledge-base/company/about.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 5 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# About Insurellm

Insurellm was founded by Avery La

In [8]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [9]:
make_messages(documents[0])

[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: company\nThe document has been retrieved from: knowledge-base/company/about.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 5 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the document:\n\n# A

In [10]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model = MODEL, messages = messages, response_format = Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [11]:
def process_document(document):
    print("\nStarting:", document["source"])

    messages = make_messages(document)

    response = completion(
        model = MODEL,
        messages = messages,
        response_format = Chunks
    )

    print("finish_reason:", response.choices[0].finish_reason)

    reply = response.choices[0].message.content
    print("reply length:", len(reply))

    try:
        doc_as_chunks = Chunks.model_validate_json(reply).chunks
    except Exception:
        print("\nFAILED DOCUMENT:", document["source"])
        print("Document length:", len(document["text"]))
        print("End of LLM response:")
        print(reply[-1000:])
        raise

    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [12]:
process_document(documents[0])


Starting: knowledge-base/company/about.md
finish_reason: stop
reply length: 3235


[Result(page_content='Introduction and Founding of Insurellm\n\nInsurellm was founded in 2015 by Avery Lancaster as an innovative insurance tech startup launching its first marketplace product.\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.', metadata={'source': 'knowledge-base/company/about.md', 'type': 'company'}),
 Result(page_content='Early Growth and Product Expansion (2015-2020)\n\nThe company experienced rapid growth, expanding its product portfolio to include various portals and platforms, reaching 200 employees and multiple offices by 2020.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a 

In [13]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [14]:
chunks = create_chunks(documents)

  0%|          | 0/76 [00:00<?, ?it/s]


Starting: knowledge-base/company/about.md


  1%|▏         | 1/76 [00:05<06:15,  5.01s/it]

finish_reason: stop
reply length: 3667

Starting: knowledge-base/company/careers.md


  3%|▎         | 2/76 [00:09<05:57,  4.83s/it]

...

finish_reason: stop
reply length: 5939

Starting: knowledge-base/products/Markellm.md


 99%|█████████▊| 75/76 [09:31<00:07,  7.98s/it]

finish_reason: stop
reply length: 5248

Starting: knowledge-base/products/Rellm.md


100%|██████████| 76/76 [09:37<00:00,  7.60s/it]

finish_reason: stop
reply length: 5110


In [15]:
print(len(chunks))

412


### Well that was easy! If a bit slow.

In the python module version, I sneakily use the multi-processing Pool to run this in parallel,
but if you get a Rate Limit Error you can turn this off in the code.

### Finally, Step 3 - save the embeddings

In [16]:
def create_embeddings(chunks):
    chroma = PersistentClient(path = DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model = embedding_model, input = texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids = ids, embeddings = vectors, documents = texts, metadatas = metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [17]:
create_embeddings(chunks)

Vectorstore created with 412 documents


# Nothing more to do here... right?

Wait! Didja think I'd forget??

In [18]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [19]:
tsne = TSNE(n_components = 2, random_state = 42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data = [go.Scatter(
    x = reduced_vectors[:, 0],
    y = reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size = 5, color = colors, opacity = 0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title = '2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title = 'x',yaxis_title = 'y'),
    width = 800,
    height = 600,
    margin=dict(r = 20, b = 10, l = 10, t = 40)
)

fig.show()

In [20]:
tsne = TSNE(n_components = 3, random_state = 42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data = [go.Scatter3d(
    x = reduced_vectors[:, 0],
    y = reduced_vectors[:, 1],
    z = reduced_vectors[:, 2],
    mode = 'markers',
    marker = dict(size = 5, color = colors, opacity = 0.8),
    text = [f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo = 'text'
)])

fig.update_layout(
    title = '3D Chroma Vector Store Visualization',
    scene = dict(xaxis_title = 'x', yaxis_title = 'y', zaxis_title = 'z'),
    width = 900,
    height = 700,
    margin = dict(r = 10, b = 10, l = 10, t = 40)
)

fig.show()

## And now - let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing

In [21]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description = "The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [22]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model = MODEL, messages = messages, response_format = RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [23]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = openai.embeddings.create(model = embedding_model, input = [question]).data[0].embedding
    results = collection.query(query_embeddings = [query], n_results = RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content = result[0], metadata = result[1]))
    return chunks

In [24]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [25]:
for chunk in chunks:
    print(chunk.page_content[:15] + "...")

Insurellm Caree...
Career Progress...
Additional HR N...
Annual Performa...
Performance Rev...
Performance and...
Emily Tran's An...
Annual Performa...
Annual Performa...
Career Progress...


In [26]:
reranked = rerank(question, chunks)

[1, 3, 8, 2, 5, 6, 4, 7, 9, 10]


In [27]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

Insurellm Caree...
Additional HR N...
Annual Performa...
Career Progress...
Performance Rev...
Performance and...
Annual Performa...
Emily Tran's An...
Annual Performa...
Career Progress...


In [28]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

In [29]:
reranked = rerank(question, chunks)

[17, 11, 13, 14, 12, 19, 18, 20, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 16]


In [30]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

In [31]:
reranked[0].page_content

"HR Record Overview of James Wilson\n\nThis section introduces James Wilson's HR record, including personal details, position, and company overview.\n\n# HR Record\n\n# James Wilson"

In [32]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [33]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [34]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context = context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [35]:
def rewrite_query(question, history = []):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model = MODEL, messages = [{"role": "system", "content": message}])
    return response.choices[0].message.content

In [36]:
rewrite_query("Who won the IIOTY award?", [])

'Who was the recipient of the IIOTY award?'

In [37]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model = MODEL, messages = messages)
    return response.choices[0].message.content, chunks

In [38]:
answer_question("Who won the IIOTY award?", [])

Who received the IIOTY award?
[1, 2, 5, 3, 4, 6, 7, 8, 11, 13, 10, 14, 12, 15, 17, 16, 18, 19, 20]


('Maxine Thompson received the IIOTY Innovator of the Year award in 2023.',
 [Result(page_content='Insurellm Career Progression (Part 3)\n\nDetails her promotion to Senior Data Engineer in 2021, major projects she led, mentorship roles, recognitions, and awards received.\n\n- **January 2021 - Present**: **Senior Data Engineer**  \n  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious IIOTY 2023 award.', metadata={'type': 'employees', 'source': 'knowledge-base/employees/Maxine Thompson.md'}),
  Result(page_content="Additional HR Notes and Initiatives\n\nDetails Maxine's participation in training, awards, mentorship, and areas for future development to enhance her professional growth.\

In [39]:
answer_question("Who went to Manchester University?", [])

Which Insurellm employees attended Manchester University?
[7, 6, 8, 20, 1, 2, 3, 4, 5, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


('Based on the information available, there is no record of any Insurellm employee having attended Manchester University.',
 [Result(page_content="Career Progression at Insurellm (Part 1)\n\nDetails Rachel Martinez's tenure from 2019 onwards as Product Manager at Insurellm, including her responsibilities and achievements.\n\n## Insurellm Career Progression\n- **March 2019 - Present:** Product Manager\n  - Leads product strategy for Carllm, the auto insurance portal\n  - Successfully launched three major feature releases that increased user engagement by 45%\n  - Manages cross-functional teams including engineering, design, and sales\n\n- **January 2017 - February 2019:** Associate Product Manager\n  - Supported product development for Marketllm marketplace\n  - Conducted user research and competitive analysis\n  - Collaborated with engineering teams on feature prioritization\n\n- **June 2015 - December 2016:** Business Analyst at TechInsure Corp\n  - Analyzed market trends and customer